In [1]:
from pathlib import Path
import json
import subprocess
import textwrap

# Update these paths for your files (must be inside this project folder).
csv_path = Path("data/extended.csv")
schema_output_path = Path("data/schema.json")

# Find project root (folder that contains docker-compose.yml).
project_root = Path.cwd().resolve()
if not (project_root / "docker-compose.yml").exists():
    for parent in project_root.parents:
        if (parent / "docker-compose.yml").exists():
            project_root = parent
            break
    else:
        raise FileNotFoundError("Could not find docker-compose.yml from current notebook path")


def resolve_under_root(path_value: Path) -> Path:
    resolved = path_value if path_value.is_absolute() else (project_root / path_value)
    resolved = resolved.resolve()
    try:
        resolved.relative_to(project_root)
    except ValueError as exc:
        raise ValueError(f"Path must be under project root: {resolved}") from exc
    return resolved


def to_container_path(host_path: Path) -> str:
    rel = host_path.relative_to(project_root)
    return f"/app/{rel.as_posix()}"


csv_host_path = resolve_under_root(csv_path)
schema_output_host_path = resolve_under_root(schema_output_path)
if not csv_host_path.exists():
    raise FileNotFoundError(f"CSV file not found: {csv_host_path}")

schema_output_host_path.parent.mkdir(parents=True, exist_ok=True)

csv_container_path = to_container_path(csv_host_path)
schema_output_container_path = to_container_path(schema_output_host_path)
job_script_host_path = project_root / "data" / "_infer_schema_job.py"
job_script_container_path = to_container_path(job_script_host_path)

# This script runs inside Docker with spark-submit.
job_script = textwrap.dedent(
    """
    import argparse
    import json
    from pyspark.sql import SparkSession

    parser = argparse.ArgumentParser()
    parser.add_argument("--csv-path", required=True)
    parser.add_argument("--output-path", required=True)
    args = parser.parse_args()

    spark = (
        SparkSession.builder
        .appName("SchemaToJson")
        .master("spark://spark-master:7077")
        .getOrCreate()
    )

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(args.csv_path)
    )

    pretty_schema_json = json.dumps(json.loads(df.schema.json()), indent=2)
    with open(args.output_path, "w", encoding="utf-8") as f:
        f.write(pretty_schema_json)

    print(pretty_schema_json)
    spark.stop()
    """
).strip() + "\n"

job_script_host_path.write_text(job_script, encoding="utf-8")

# 1) Start Spark master/workers.
subprocess.run(
    [
        "docker",
        "compose",
        "up",
        "-d",
        "spark-master",
        "spark-worker-1",
        "spark-worker-2",
        "spark-worker-3",
    ],
    cwd=str(project_root),
    check=True,
)

# 2) Run schema inference in Docker.
subprocess.run(
    [
        "docker",
        "compose",
        "--profile",
        "dq",
        "run",
        "--rm",
        "dq-runner",
        "/opt/spark/bin/spark-submit",
        "--master",
        "spark://spark-master:7077",
        "--deploy-mode",
        "client",
        job_script_container_path,
        "--csv-path",
        csv_container_path,
        "--output-path",
        schema_output_container_path,
    ],
    cwd=str(project_root),
    check=True,
)

# 3) Show the generated schema JSON from host path.
pretty_schema_json = json.loads(schema_output_host_path.read_text(encoding="utf-8"))
print("Inferred schema JSON:")
print(json.dumps(pretty_schema_json, indent=2))
print(f"Schema saved to: {schema_output_host_path}")


Inferred schema JSON:
{
  "fields": [
    {
      "metadata": {},
      "name": "SRC_SYS_COD",
      "nullable": true,
      "type": "string"
    },
    {
      "metadata": {},
      "name": "CTL_ARE_COD",
      "nullable": true,
      "type": "string"
    },
    {
      "metadata": {},
      "name": "MAT_TYP_COD",
      "nullable": true,
      "type": "string"
    },
    {
      "metadata": {},
      "name": "MAT_COD",
      "nullable": true,
      "type": "string"
    },
    {
      "metadata": {},
      "name": "PLT_COD",
      "nullable": true,
      "type": "string"
    },
    {
      "metadata": {},
      "name": "VDR_COD",
      "nullable": true,
      "type": "integer"
    },
    {
      "metadata": {},
      "name": "PUR_DOC_NUM",
      "nullable": true,
      "type": "long"
    },
    {
      "metadata": {},
      "name": "PUR_DOC_ITM",
      "nullable": true,
      "type": "integer"
    },
    {
      "metadata": {},
      "name": "PUR_DOC_LIN_NUM",
      "nullable": true,
 